In [ ]:
from src.utils.data_utils import build_dataloaders
from src.utils.train_utils import ReadType as ReadType, TASK_META_MAP, BATCH_ADAPTERS
from src.config import Config
from src.tasks import Tasks
from src.model import WM_Model,Memory_Components

from tqdm.auto import tqdm

import torch

import os

import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from src.config import Config
def load_config_from_checkpoint(config_dict):
    
    new_config = Config.__new__(Config)
    for key, value in config_dict.items():
        setattr(new_config, key, value)
            
    return new_config

def _to_device(batch: dict, device):

    """
    Moves all tensor values in the batch dict to the trainer device.
    Keeps non-tensors (ints, strings, etc.) unchanged.
    """
    out = {}
    for k, v in batch.items():
        out[k] = v.to(device) if torch.is_tensor(v) else v

    return out



In [ ]:
def perform_neural_analysis(model_names):

    loaders_exists = False

    for model_name in model_names:

        checkpoint_path = os.path.join(f"WM_Bench/{model_name}/checkpoints", "best.pt")
        checkpoint_data = torch.load(checkpoint_path, weights_only= False)

        # set up config
        config_dict = checkpoint_data["config"]
        config = load_config_from_checkpoint(config_dict)
        config.train_config.batch_size = 1
        task_list = config.task_config.task_list

        # set up device
        device = "cuda" if torch.cuda.is_available() else "cpu"

        # set up model 
        model = WM_Model(config, device).to(device)
        model.load_state_dict(checkpoint_data['model_state_dict'])

        # set up test loaders
        if not loaders_exists:
            loaders = build_dataloaders(config)
            test_loaders = {t: loaders[t]["test"] for t in config.task_config.task_list}
            per_task_loaders = [test_loaders[task] for task in config.task_config.task_list]
            multitask_iterator = zip(*per_task_loaders)

            loaders_exists = True


        # run the batch 
        model.eval()
        # mem_output_dict to store results 
        mem_output_dict = {task: [] for task in task_list}    

        with torch.no_grad():
            for step_index, multitask_batch in tqdm(enumerate(multitask_iterator)):
                for task_idx, task in enumerate(config.task_config.task_list):
                    raw_task_batch = multitask_batch[task_idx]

                    # Task Meta 
                    meta = TASK_META_MAP[task]
                    is_multilabel = bool(meta.get("multi_label", False))
                    loss_type = meta["loss_type"] 
                    read_type = meta["read_type"] 


                    # For TAIL Tasks meta tells us where k comes from: "set_size" or "list_length"
                    k_from = meta.get("k_from", None)
                    pad_value = meta.get("pad_value", None) 

                    # First we need to normalize the raw_batch
                    batch = BATCH_ADAPTERS[task](batch = raw_task_batch)

                    # The most common fields are stim (img_seq), resp (target), and seq_len (a tensor of sequence lengths for each stim in the batch)
                    batch = _to_device(batch, device)
                    stim = batch["img_seq"]
                    resp = batch["gt"]
                    seq_len = batch["seq_len"]

                    batch_size = stim.shape[0]

                    # We only want the hidden state 
                    output, mem_output, mem_h_n, projection_output, cnn_output = model(stim, task, seq_len)

                    hidden_states = mem_output[0].detach().cpu()

                    seq_len = seq_len[0].item()

                    # Remove padding 
                    if read_type == ReadType.TAIL:
                        k_from_val = batch[k_from.value][0].item()
                        valid_hidden = hidden_states[:seq_len - k_from_val, :]

                    elif read_type == ReadType.FINAL:
                        valid_hidden = hidden_states[:seq_len - 1, :]

                    elif read_type == ReadType.SEQUENCE:
                        valid_hidden = hidden_states[:seq_len, :]

                    mem_output_dict[task].append(valid_hidden.numpy())

        merged_mem_output = {}
        cd_trials = []

        for task, trials in mem_output_dict.items():
            # Group all Change Detection variants into one 'CD_Task'
            if "CHANGE_DETECTION" in task.name:
                cd_trials.extend(trials)
            else:
                # Keep other tasks as they are
                merged_mem_output[task] = trials

        merged_mem_output["CD_Task"] = cd_trials
        final_task_names = [t.name for t in task_list if "CHANGE_DETECTION" not in t.name] + ["CD_Task"]

        
        task_variance_map = {}

        for task, trials_list in mem_output_dict.items():
            # We need to calculate the variance on the time axis of each tiral 
            # trial shape --> (T, units)
            # variance shape --> (units, )
            trial_vars = [np.var(trial, axis = 0) for trial in trials_list]

            # average accross 
            task_variance_map[task] = np.mean(trial_vars, axis=0)

        variance_matrix = np.array([task_variance_map[task] for task in task_list])

        # Normalize each column by its max variance accross any task
        normalized_matrix = variance_matrix / (np.max(variance_matrix, axis= 0) + 1e-10)

        clustering_input = normalized_matrix.T

        num_cluster_range = []
        silhouette_scores = []

        for n in range(2, 100):
            km = KMeans(n_clusters= n, random_state=config.optimization_config.seed).fit(clustering_input)
            score = silhouette_score(clustering_input, km.labels_)
            num_cluster_range.append(n)
            silhouette_scores.append(score)

        best_n = num_cluster_range[np.argmax(silhouette_scores)]
        print(f"Optimal Clusters: {best_n}")

        kmeans = KMeans(n_clusters=best_n, random_state=config.optimization_config.seed).fit(clustering_input)
        sorted_indices = np.argsort(kmeans.labels_)
        sorted_matrix = clustering_input[sorted_indices]

        plt.figure(figsize = (15, 8))
        sns.heatmap(sorted_matrix.T, cmap="hot", yticklabels=final_task_names)
        plt.xlabel("Units (Sorted by Cluster)")
        plt.ylabel("Tasks")
        plt.title(f"Functional Clustering of Memory Units - {model_name}")
        plt.show()

In [ ]:
model_names = ["GRU-256", "LSTM-256", "RNN-256", "TRF-256"]
perform_neural_analysis(model_names)